# Clinical DataLakehouse Modernization - Pipeline Audit

## Purpose
This notebook orchestrates the execution of the Bronze, Silver, and Gold data pipeline layers, logs their execution status, and captures audit information for monitoring and traceability.

## Completed Steps
- Loaded configuration settings.
- Created the audit schema and audit log table using Delta Lake.
- Defined notebook paths for each pipeline layer (Bronze, Silver, Gold).
- Executed each pipeline layer notebook and captured execution details (status, record counts, errors, timestamps).
- Appended audit records to the audit log table.
- Displayed record counts for each layer.
- Displayed the audit log for review.

In [0]:
    %run ./config

##### Creating the audit schema

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {cfg['catalog']}.{cfg['schema_audit']}
""")

##### Create audit table 

In [0]:
%skip
%sql
DROP TABLE IF EXISTS
clinical_datalakehouse_modernization.audit.pipeline_audit_log;

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {cfg['log_table']}
(
    layer_name STRING,
    table_name STRING,
    record_count BIGINT,
    status STRING,
    error_message STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    execution_time_seconds DOUBLE,
    batch_id STRING
)
USING DELTA
""")

##### Importing the required libraries

In [0]:
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType, DoubleType

##### Notebook Paths

In [0]:
notebooks = [

    ("Bronze",
     "/Workspace/Users/takiakio09@gmail.com/Clinical_DataLakehouse_Modernization/medallion architecture/01_Bronze_Load"),

    ("Silver",
     "/Workspace/Users/takiakio09@gmail.com/Clinical_DataLakehouse_Modernization/medallion architecture/02_Silver_Load"),

    ("Gold",
     "/Workspace/Users/takiakio09@gmail.com/Clinical_DataLakehouse_Modernization/medallion architecture/03_Gold_Load")

]

##### Audit Execution loop

In [0]:
# Define the schema for the audit log
audit_schema = StructType([
    StructField("layer_name", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("record_count", LongType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("execution_time_seconds", DoubleType(), True),
    StructField("batch_id", StringType(), True)
])

# Define table names for each layer from config
layer_tables = {
    "Bronze": cfg['bronze_table'],
    "Silver": cfg['silver_table'],
    "Gold": cfg['gold_table']
}

for layer_name, notebook_path in notebooks:

    start_time = datetime.now()

    status = "SUCCESS"
    error_message = None
    table_name = layer_tables.get(layer_name)
    record_count = None

    try:

        dbutils.notebook.run(
            notebook_path,
            0
        )

        # Query record count after successful execution
        if table_name:
            record_count = spark.table(table_name).count()

    except Exception as e:

        status = "FAILED"
        error_message = str(e)

    end_time = datetime.now()

    execution_time = (
        end_time - start_time
    ).total_seconds()

    audit_record = [
        (
            layer_name,
            table_name,
            record_count,
            status,
            error_message,
            start_time,
            end_time,
            execution_time,
            str(start_time)  # batch_id
        )
    ]

    audit_df = spark.createDataFrame(
        audit_record,
        schema=audit_schema
    )

    audit_df.write.format("delta").mode("append").saveAsTable(
        cfg['log_table']
    )

    print(
        f"{layer_name} Completed : {status} | Table: {table_name} | Records: {record_count:,}" if record_count else f"{layer_name} Completed : {status}"
    )

##### Capture record counts

In [0]:
# Capture record counts from config
try:
    bronze_count = spark.table(cfg['bronze_table']).count()
    print("Bronze :", bronze_count)
except Exception as e:
    print(f"Bronze : Table not found - {cfg['bronze_table']}")

try:
    silver_count = spark.table(cfg['silver_table']).count()
    print("Silver :", silver_count)
except Exception as e:
    print(f"Silver : Table not found - {cfg['silver_table']}")

try:
    gold_count = spark.table(cfg['gold_table']).count()
    print("Gold :", gold_count)
except Exception as e:
    print(f"Gold : Table not found - {cfg['gold_table']}")

##### View audit logs

In [0]:
# View audit logs from config
display(
    spark.table(cfg['log_table'])
)